In [50]:
pip install groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =============================
# IMPORTS
# =============================
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

# =============================
# INIT LLM CLIENT
# =============================
client = Groq(api_key="please enter your api key") # 🔴 replace with your key


# =============================
# LOAD DATA
# =============================
with open("products.json", "r", encoding="utf-8") as f:
    products = json.load(f)


# =============================
# BUILD SEARCH TEXT
# =============================
def build_text(p):
    return (
        f"{p.get('name','')} "
        f"{p.get('description','')} "
        f"{p.get('shortDescription','')} "
        f"{p.get('brand','')}"
    ).lower()


# =============================
# LOAD MODEL
# =============================
model = SentenceTransformer('all-MiniLM-L6-v2')


# =============================
# PREPROCESS PRODUCTS
# =============================
for p in products:
    p["searchText"] = build_text(p)
    p["embedding"] = model.encode(p["searchText"])


# =============================
# BUILD FAISS INDEX
# =============================
embedding_matrix = np.array([p["embedding"] for p in products]).astype("float32")
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)




# =============================
# COSINE SIMILARITY
# =============================
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# =============================
# NORMALIZE SCORE
# =============================
def normalize_score(raw_score):
    return 1 / (1 + np.exp(-raw_score))


# =============================
# SCORE LABEL
# =============================
def get_score_label(score):
    if score < 0.4:
        return "Weak match"
    elif score < 0.7:
        return "Good match"
    elif score < 0.9:
        return "Strong match"
    else:
        return "Very strong match 🔥"
    
    
# =============================
# 🔥 LLM: QUERY UNDERSTANDING
# =============================
def understand_query_llm(user_query):
    try:
        prompt = f"""
        
        You are an AI assistant for an e-commerce recommendation system.

        Your task is to understand the user's intent and convert it into a clear, standardized shopping occasion query.

        Rules:
        - Focus on intent, not exact words
        - Normalize into common shopping occasions
        - Keep output short (2–4 words)
        - Always return a product-focused query 
        - Do NOT explain anything
        - Do NOT return multiple options
        
        


        Query: {user_query}
        Return ONLY the normalized query.
        """

        response = client.chat.completions.create(
            model= "llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20
        )

        return response.choices[0].message.content.strip().lower()

    except:
        return user_query.lower()


# =============================
# 🔥 LLM: GENERATE EXPLANATION
# =============================


def generate_reason_llm(product, query):
    
        prompt = f"""
        User query: {query}
        Product: {product['name']}
        Description: {product.get('shortDescription','')}

        Explain in ONE short sentence why this product is relevant.
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",   
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=40
        )

        return response.choices[0].message.content.strip()

    


# =============================
# FAISS SEARCH
# =============================
def search_faiss(query, top_k=50):
    query_embedding = model.encode(query).astype("float32")
    distances, indices = index.search(np.array([query_embedding]), top_k)
    return [products[i] for i in indices[0]]


# =============================
# MAIN RECOMMENDATION FUNCTION
# =============================
def recommend_by_occasion(user_input, top_k=10):

    original_query = user_input.lower()
    user_input = understand_query_llm(original_query)

    # 🔥 STEP 1: FAST SEARCH
    candidates = search_faiss(user_input, top_k=50)

    query_embedding = model.encode(user_input)

    results = []

    for p in candidates:

        if not p.get("isValid", True):
            continue

        # Semantic similarity
        sim_score = cosine_similarity(query_embedding, p["embedding"])

        # 3. Occasion boost
        occasion_score = 0

        for tag in p.get("occasionTags", []):
            if tag in user_input:
                occasion_score += 0.3   # strong boost

        # 4. Tag boost
        tag_score = 0

        for tag in p.get("tags", []):
            if tag in user_input:
                tag_score += 0.1

        raw_score = sim_score + occasion_score + tag_score

        results.append((raw_score, p))

    # Sort
    results.sort(key=lambda x: x[0], reverse=True)

    # =============================
    # FORMAT OUTPUT
    # =============================
    output = []

    for raw_score, p in results[:top_k]:

        score = normalize_score(raw_score)
        label = get_score_label(score)

        desc = p.get("shortDescription") or p.get("description", "")

        # 🔥 LLM EXPLANATION
        reason = generate_reason_llm(p, user_input)

        output.append({
            "title": p.get("name"),

            # Your required format
            "category": f'{p.get("categoryId")} ("{p.get("name")}")',

            "description": desc[:120] + "...",
            "image": p.get("mainImage"),

            "score": round(score, 2),
            "match": label,

            # 🔥 NEW FIELD (VERY IMPORTANT)
            "reason": reason
        })

    return output


# =============================
# TEST
# =============================
if __name__ == "__main__":

    query = "gift for wife"

    results = recommend_by_occasion(query)

    for r in results:
        print("\n---------------------")
        print("Title:", r["title"])
        print("Category:", r["category"])
        print("Description:", r["description"])
        print("Image:", r["image"])
        print("Score:", r["score"])
        print("Match:", r["match"])
        print("Reason:", r["reason"])
       


---------------------
Title: Housewarming Gifts
Category: cml258vol00eqma01a56hvc3y ("Housewarming Gifts")
Description: Personal Celebrations honor life’s most cherished moments—birthdays, anniversaries, engagements, weddings, and milestone...
Image: https://s3aws.emilo.in/products/1769855880273-7f544a95-29fb-4d80-8e43-1d78f877e56f.png
Score: 0.63
Match: Good match
Reason: This product is relevant because a housewarming gift can be a thoughtful gift for a wife, especially if you're moving into a new home together.

---------------------
Title: Gift Wrapping
Category: cml0ug1d6003to201xuwjkmcn ("Gift Wrapping")
Description: Our Gift Accessories are designed to elevate every gift with thoughtful detail and refined style. From elegant wrapping ...
Image: https://s3aws.emilo.in/products/1769851348474-322c67f6-af06-4ab9-b9d4-4205721d8b0a.png
Score: 0.61
Match: Good match
Reason: This product is relevant because it helps make a gift for a wife more special and elegantly presented.

--------